# Week 2 briefing: DoU, label gaps, bbox CNN

Last week's downtown LORO was almost entirely an urban-core number (about 98.7% urban). Expanding the boxes brings peri-urban and rural buildings. Overture subtype fill falls 35% to 20% to 14% (urban to peri-urban to rural). Binary LORO F1 drops only mildly (0.78 to 0.76 to 0.74). A footprint bbox CNN on AlphaEarth grids lifts binary LORO from about 0.70 (centroid LR) to about 0.75. Missing use tags look more like an OSM vs ML coverage gap than a large residential-only bias on FEMA / GHS-OBAT checks.

Setup: GHSL GHS-SMOD Degree of Urbanization, hinterland rings around the trial metros, Track-A Overture where tagged. CNN uses enclosing AE grid patches (`datasets/building_bbox_grid/`). Figures under `outputs/week2_*`.


## 0. Why this week

We asked whether RES / NRES behaves the same in the city, suburbs, and countryside, and whether missing Overture tags are biased. We also tested whether pooling AlphaEarth cells over each footprint beats a single centroid.


## 1. Downtown sample was almost entirely urban

Notebook 02's tight metro cores are 98.7% urban (3,554 of 3,600). Peri-urban and rural are essentially absent. Urban-core F1 cannot answer the urban to rural question until the sample includes hinterland.

![Downtown boxes are almost entirely urban](../../outputs/week2_ghsl_dou/core_is_urban.png)


## 2. Expanding the box brings suburbs and countryside

The small black rectangle is last week's downtown; the larger area is this week's hinterland. Gold (peri-urban) and green (rural) points show up outside the core. Cities differ: LA's expanded box stays mostly urban; Denver and Seattle show clearer urban to rural transitions.

![Where the hinterland sample sits](../../outputs/week2_ghsl_dou/hinterland_maps_by_dou.png)

![US core vs hinterland](../../outputs/week2_ghsl_dou/us_core_vs_hinterland.png)


## 3. Overture tags get rarer outside the city

Subtype fill falls 35% to 20% to 14% (urban to peri-urban to rural). Among tagged buildings, residential still dominates (about 84-92%). Unlabeled footprints stay excluded.

| DoU | Footprints | Fill rate | Tagged n | % residential among tagged |
|---|---:|---:|---:|---:|
| Urban | 7,692 | 35.0% | 2,692 | 91% |
| Peri-urban | 2,098 | 20.3% | 425 | 92% |
| Rural | 1,010 | 14.0% | 141 | 84% |

![Tags get rarer as you leave the city](../../outputs/week2_ghsl_dou/fill_rate_story.png)

![Fill rate and mix by DoU](../../outputs/week2_ghsl_dou/fill_rate_and_mix_by_dou.png)


## 4. Classifier holds with a mild urban to rural drop

Same model family as the Week 1 trial (logistic regression on 64-D AlphaEarth, LORO), with hinterland labels added:

| DoU | n (OOF) | Macro F1 (train all DoU) | Macro F1 (train urban only) |
|---|---:|---:|---:|
| Urban | 5,092 | 0.78 | 0.78 |
| Peri-urban | 460 | 0.76 | 0.75 |
| Rural | 208 | 0.74 | 0.71 |

Rural n is small; treat rural F1 as directional. City holdout still varies a lot.

![F1 by DoU and city](../../outputs/week2_ghsl_dou/f1_by_dou_and_city.png)

![F1 by DoU](../../outputs/week2_ghsl_dou/f1_by_dou.png)


## 5. Missingness: coverage gap more than class bias

Question: are unlabeled buildings systematically more residential (or less) than labeled ones?

On the hinterland sample (n = 10,800, about 30% labeled), OSM source fill is about 37% while Microsoft ML footprints contribute geometry with essentially 0% use tags. FEMA matches about 85% of buildings; GHS-OBAT about 96%. Where external residential proxies exist, unlabeled vs labeled deltas are usually a few points, not a one-sided wipeout. Rule of thumb we used: similar P(res) implies a coverage gap; a large delta would force sampling / city-exclusion rules.

![Sources vs fill](../../outputs/week2_missingness/sources_vs_fill.png)

![Missingness bias delta](../../outputs/week2_missingness/missingness_bias_delta.png)

![FEMA bias by city](../../outputs/week2_missingness/fema_bias_by_city.png)

![GHS-OBAT bias by city](../../outputs/week2_missingness/obat_bias_by_city.png)

Notebook: `notebooks/week2/01_overture_missingness_bias.ipynb`.


## 6. Footprint bbox CNN beats centroid LR

Enclosing AlphaEarth grid over each building bbox, small CNN, LORO on the multi-class US set (binary rollup of SF/MF vs rest).

| Task | Centroid LR | Bbox CNN | Delta |
|---|---:|---:|---:|
| Binary RES/NRES | 0.70 | 0.75 | +0.05 |
| 10-class | 0.18 | 0.23 | +0.06 |

Gains concentrate in harder metros (Denver, Boston). Atlanta binary is one of the few CNN underperforms.

![Binary LORO CNN vs LR](../../outputs/week2_cnn_bbox/binary_loro_cnn_vs_lr.png)

![Multiclass LORO CNN vs LR](../../outputs/week2_cnn_bbox/multiclass_loro_cnn_vs_lr.png)

![UMAP CNN by residential](../../outputs/week2_cnn_bbox/umap_cnn_by_residential.png)

![Within-region residential separability](../../outputs/week2_cnn_bbox/within_region_resi_separability.png)


### Embedding view (per metro)

OOF CNN pooled features in UMAP, one panel per holdout metro. The boundary panels draw a logistic decision surface on the 2-D UMAP coords so you can see where RES and NRES still mix.

![UMAP CNN by region panels, residential](../../outputs/week2_cnn_bbox/umap_cnn_by_region_panels_residential.png)

![UMAP CNN by region panels with LR boundary](../../outputs/week2_cnn_bbox/umap_cnn_by_region_panels_residential_boundary.png)

Single-metro panels: `outputs/week2_cnn_bbox/umap_region_panels/umap_<region>_residential_boundary.png`.

Notebook: `notebooks/week2/02_cnn_bbox_grid_training.ipynb`. Map UI: `python scripts/serve_map_overlay.py`.


## 7. GHSL Built-C vs AlphaEarth (same GT)

GHSL GHS-BUILT-C FUN is the frozen baseline we score against Track-A Overture. Building score = majority of 10 m cells under the footprint. Pixel score = same raster on the GHSL grid (see Week 3 §2 for the pixel paint).

| Method | Macro-F1 |
|---|---:|
| Polygon AE bbox CNN (this week's model) | ~0.75 |
| Polygon AE centroid LR | ~0.70 |
| GHSL Built-C (building) | ~0.67 |
| Pixel AE (Week 3 binary) | ~0.56–0.57 |
| Pixel GHSL | ~0.51 |

![Building AE CNN beats GHSL](../../outputs/week3_model_comparison/figures/01_building_ae_beats_ghsl.png)

![Functional bars: CNN vs GHSL at building and pixel](../../outputs/validation/figures/functional_comparison_bars.png)

AE wins on macro-F1. GHSL is high-recall / low-precision on RES (recall ~0.96, precision ~0.46 at building level). Tables: `outputs/validation/comparison_table.md`.


## 8. Takeaways

| Decision | Evidence |
|---|---|
| Report LORO overall and by DoU | Downtown-only F1 hid hinterland |
| Keep hinterland (or another rural draw) in samples | Core was 98.7% urban |
| Label availability is the bigger DoU gap than F1 collapse | Fill 35 to 14%; F1 0.78 to 0.74 |
| Prefer bbox / pooled AE over single centroid when footprints exist | Binary +0.05 macro-F1 |
| Treat unlabeled as excluded, not negatives | OSM vs ML coverage story |

Related notebooks: `09_week2_ghsl_degree_of_urbanization.ipynb`, `week2/01_overture_missingness_bias.ipynb`, `week2/02_cnn_bbox_grid_training.ipynb`. Docs: `docs/week2_ghsl_dou_findings.md`.


## Appendix: load live numbers


In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path('.').resolve()
for _ in range(3):
    if (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent

paths = {
    'GHSL DoU': ROOT / 'outputs/week2_ghsl_dou/week2_snapshot.json',
    'missingness bias': ROOT / 'outputs/week2_missingness/week2_missingness_snapshot.json',
    'bbox CNN': ROOT / 'outputs/week2_cnn_bbox/cnn_bbox_snapshot.json',
    'GHSL vs AE validation': ROOT / 'outputs/validation/summary.json',
}
for name, p in paths.items():
    display(Markdown(f'### {name}'))
    if not p.exists():
        print('missing:', p)
        continue
    d = json.loads(p.read_text())
    # drop long file lists
    slim = {k: v for k, v in d.items() if k != 'outputs'}
    print(json.dumps(slim, indent=2)[:2500])
